# AccessMap: Modeling & Retrieval (PMR + Housing)

## Project Description
This is notebook 3 of 3. Notebooks 1 and 2 cleaned PMR (Amsterdam
sidewalks) and Housing (US accessibility labels). Both are saved in
`data/processed/`. This notebook builds the models.

## Why we don't merge PMR and Housing
We checked the coordinates. PMR is Amsterdam only (EPSG:28992). Housing
is spread across the whole US. No overlap. A spatial join would match
Amsterdam sidewalks to houses thousands of miles away. That's noise,
not signal. So we run the same 3 algorithms on each dataset separately,
then compare results at the end.

## Dataset
- **PMR:** `data/processed/pmr_clean.gpkg` (from notebook 01)
- **Housing:** `data/processed/housing_clean.csv` (from notebook 02)

## Project Roadmap

| Step | What we do |
|------|------------|
| 1 | Load both clean datasets |
| 2 | PMR: build an accessibility label, pick features |
| 3 | PMR: Random Forest |
| 4 | Housing: pick features, Random Forest |
| 5 | Compare both Random Forest models |
| 6 | ModernBERT embeddings (Housing address text) |
| 7 | FAISS similarity search (within each dataset) |
| 8 | Export models, embeddings, index |

## Note on the PMR label
PMR's real `wheelchair_accessible` column only has 50 rows filled in,
out of 72,274. All 50 are transit stops, not sidewalks. Too few rows
to train on. So for the 72k sidewalk rows we build our own label,
`pmr_accessible`, from width and curb height. This is our own rule, not
a verified label like Housing's `sidewalk_ok`. Keep this in mind for
the bias write-up.


In [11]:
# Step 1: Load Clean Data (PMR + Housing)
# Just load both files from notebook 01/02. No cleaning here.
import pandas as pd
import geopandas as gpd

PMR_PATH = '../data/processed/pmr_clean.gpkg'
HOUSING_PATH = '../data/processed/housing_clean.csv'

pmr = gpd.read_file(PMR_PATH)
housing = pd.read_csv(HOUSING_PATH)

print(f"PMR shape: {pmr.shape[0]:,} rows x {pmr.shape[1]} columns")
print(f"Housing shape: {housing.shape[0]:,} rows x {housing.shape[1]} columns")

print(f"\nPMR CRS: {pmr.crs}")
minx, miny, maxx, maxy = pmr.total_bounds
print(f"PMR bounding box (meters, EPSG:28992): "
      f"x=[{minx:,.0f}, {maxx:,.0f}], y=[{miny:,.0f}, {maxy:,.0f}]")
print(f"  -> roughly a {maxx-minx:,.0f}m x {maxy-miny:,.0f}m area (Amsterdam)")

print("\nHousing lat/long range:")
print(housing[['lat', 'long']].describe().loc[['min', 'max']].round(2))

# CHANGED: plain-English summary of scale, plus the actual point of
# this step -- confirming no overlap
lat_span = housing['long'].max() - housing['long'].min()
print(f"  -> spans {lat_span:,.0f} degrees longitude, roughly the width "
      f"of the entire US")
print(f"\nNo overlap: PMR is one small area in Amsterdam (meters). "
      f"Housing spans a continent (degrees). Keep the two pipelines separate.")

PMR shape: 72,274 rows x 19 columns
Housing shape: 10,104 rows x 19 columns

PMR CRS: EPSG:28992
PMR bounding box (meters, EPSG:28992): x=[114,084, 122,716], y=[484,488, 487,978]
  -> roughly a 8,632m x 3,491m area (Amsterdam)

Housing lat/long range:
       lat    long
min  25.64 -123.35
max  77.55  166.16
  -> spans 290 degrees longitude, roughly the width of the entire US

No overlap: PMR is one small area in Amsterdam (meters). Housing spans a continent (degrees). Keep the two pipelines separate.


## Step 1 Summary - Load Clean Data

### What we did:
- Loaded `pmr_clean.gpkg` and `housing_clean.csv`. Both are already
  clean from notebooks 01 and 02.

### What we found:
- PMR: 72,274 rows, Amsterdam only (~8,632m x 3,491m area).
- Housing: 10,104 rows, spread across the US (spans 290 degrees
  longitude — roughly the width of the entire continent).
- Both now have 19 columns (18 original + a confidence flag from
  notebook 01/02: `low_confidence_width` for PMR,
  `low_confidence_residential` for Housing).
- No geographic overlap. PMR is one small area measured in meters.
  Housing spans a continent measured in degrees. Confirms we keep the
  two pipelines separate.

### What this means:
- No spatial join is possible or appropriate here. Any comparison
  between PMR and Housing has to happen at the model-result level
  (Step 5), not the row level.

### Next step:
-> Step 2: PMR Accessibility Label + Features


In [12]:
# Step 2: PMR - Accessibility Label + Features
# Build the pmr_accessible label, then pick and encode features.
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import LabelEncoder

pmr = gpd.read_file('../data/processed/pmr_clean.gpkg')

# ---------------------------------------------------------
# 2.1 Our accessibility rule
# ---------------------------------------------------------
width_ok = pmr['obstacle_free_width_float'] >= 0.9
curb_ok = pmr['curb_height_max'].isna() | (pmr['curb_height_max'] <= 0.06)
pmr['pmr_accessible'] = (width_ok & curb_ok).astype(int)

label_counts = pmr['pmr_accessible'].value_counts(normalize=True)
print("Label balance:")
print(f"  Accessible (1):     {label_counts[1]:.1%}  ({(pmr['pmr_accessible']==1).sum():,} rows)")
print(f"  Not accessible (0): {label_counts[0]:.1%}  ({(pmr['pmr_accessible']==0).sum():,} rows)")

# ---------------------------------------------------------
# 2.2 Pick features
# ---------------------------------------------------------
pmr_features = ['length', 'obstacle_free_width_float', 'crossing', 'width_fill']
pmr_model_df = pmr[pmr_features + ['pmr_accessible']].copy()

# CHANGED: name the encoder so Step 8 can save it
crossing_encoder = LabelEncoder()
pmr_model_df['crossing'] = crossing_encoder.fit_transform(
    pmr_model_df['crossing'].astype(str)
)
pmr_model_df = pmr_model_df.fillna(pmr_model_df.median(numeric_only=True))

print(f"\nFeature table shape: {pmr_model_df.shape[0]:,} rows x {pmr_model_df.shape[1]} columns")
pmr_model_df.head()

Label balance:
  Accessible (1):     80.8%  (58,394 rows)
  Not accessible (0): 19.2%  (13,880 rows)

Feature table shape: 72,274 rows x 5 columns


,length,obstacle_free_width_float,crossing,width_fill,pmr_accessible
0,1.99,1.6,0,0.0,1
1,1.99,1.6,0,0.0,1
2,1.99,1.6,0,0.0,1
3,1.99,1.6,0,0.0,1
4,1.99,1.6,0,0.0,1


## Step 2 Summary - PMR Accessibility Label + Features

### What we did:
- Built `pmr_accessible` from width + curb height. The real
  `wheelchair_accessible` column is too small to use (see intro note).
- Picked 4 features with low missing data: `length`,
  `obstacle_free_width_float`, `crossing`, `width_fill`.
- Turned `crossing` (Yes/No) into 0/1. Filled the rest with the
  column median.

### What we found:
- Label balance: 80.8% accessible (58,394 rows), 19.2% not accessible (13,880 rows).
- Feature table: 72,274 rows x 5 columns (4 features + label).

### What this means:
- Good enough balance for Random Forest with `class_weight='balanced'`.
- The label comes from width and curb height. So expect
  `obstacle_free_width_float` to dominate feature importance in Step
  3. That's just our rule showing up again, not a new finding. Say
  this clearly in the Step 3 write-up so it's not overstated.

### Next step:
-> Step 3: PMR Random Forest


In [13]:
# Step 3: PMR - Random Forest
# Train and check Random Forest on the pmr_accessible label.
import pandas as pd
import geopandas as gpd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

pmr = gpd.read_file('../data/processed/pmr_clean.gpkg')
width_ok = pmr['obstacle_free_width_float'] >= 0.9
curb_ok = pmr['curb_height_max'].isna() | (pmr['curb_height_max'] <= 0.06)
pmr['pmr_accessible'] = (width_ok & curb_ok).astype(int)

# BEFORE: pmr_features = ['length', 'obstacle_free_width_float', 'crossing', 'width_fill']
# CHANGED: added curb_height_max and a missing flag. Why: precision on
# not_accessible was 0.55. We tried moving the decision threshold first
# (0.5 -> 0.65). Didn't work. Only 14 of 14,455 predictions sat between
# 0.35 and 0.5, so the threshold had nothing to move. The model is
# confidently wrong, not unsure. That points at features, not the cutoff.
# curb_height_max is half of the label rule but the model never saw it.
# 74% of it is missing (see notebook 01), so we add a missing flag too
# instead of just filling the median and hiding it.
pmr['curb_height_missing'] = pmr['curb_height_max'].isna().astype(int)

pmr_features = ['length', 'obstacle_free_width_float', 'crossing', 'width_fill',
                 'curb_height_max', 'curb_height_missing']
pmr_model_df = pmr[pmr_features + ['pmr_accessible']].copy()

# CHANGED: reuse the same crossing_encoder from Step 2, don't create a new one
pmr_model_df['crossing'] = crossing_encoder.transform(
    pmr_model_df['crossing'].astype(str)
)
pmr_model_df = pmr_model_df.fillna(pmr_model_df.median(numeric_only=True))

X = pmr_model_df[pmr_features]
y = pmr_model_df['pmr_accessible']
X_train_pmr, X_test_pmr, y_train_pmr, y_test_pmr = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_pmr = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced', random_state=42
)
rf_pmr.fit(X_train_pmr, y_train_pmr)

# Back to the plain predict() call. The threshold change didn't help
# (see comment above), so no reason to keep the extra code.
pmr_pred = rf_pmr.predict(X_test_pmr)

acc = accuracy_score(y_test_pmr, pmr_pred)
baseline = y_test_pmr.value_counts(normalize=True).max()

print("PMR Random Forest")
print(f"Accuracy:              {acc:.1%}")
print(f"Majority-class baseline: {baseline:.1%}  (always guessing 'accessible')")
print(f"Improvement over baseline: {(acc - baseline)*100:+.1f} percentage points")
print()
print(classification_report(y_test_pmr, pmr_pred, target_names=['not_accessible', 'accessible']))

pmr_importance = pd.Series(rf_pmr.feature_importances_, index=pmr_features).sort_values(ascending=False)
print("\nFeature importance (higher = more influence on the prediction):")
for feat, score in pmr_importance.items():
    print(f"  {feat:<28} {score:.1%}")


PMR Random Forest
Accuracy:              100.0%
Majority-class baseline: 80.8%  (always guessing 'accessible')
Improvement over baseline: +19.2 percentage points

                precision    recall  f1-score   support

not_accessible       1.00      1.00      1.00      2776
    accessible       1.00      1.00      1.00     11679

      accuracy                           1.00     14455
     macro avg       1.00      1.00      1.00     14455
  weighted avg       1.00      1.00      1.00     14455


Feature importance (higher = more influence on the prediction):
  obstacle_free_width_float    42.4%
  curb_height_max              37.6%
  curb_height_missing          8.6%
  width_fill                   5.3%
  crossing                     3.5%
  length                       2.6%


## Step 3 Summary - PMR Random Forest

### What we did:
- Split 80/20, trained a Random Forest (`n_estimators=200,
  max_depth=10, class_weight='balanced'`).
- First try: moved the not_accessible threshold from 0.5 to 0.65 to
  fix the low precision (0.55). Didn't work. Only 14 of 14,455 test
  predictions sat between 0.35 and 0.5, so the threshold had almost
  nothing to move. The model is confidently wrong on some rows, not
  unsure.
- Second try: added `curb_height_max` and a `curb_height_missing`
  flag as features. Curb height is half of the label rule, but the
  model never saw it. 74% of it is missing, so the missing flag
  matters too.

### What we found:
- Old: accuracy 84.2% vs. baseline 80.8% (+3.4pp). not_accessible
  recall 1.00, precision 0.55.
- New: accuracy 100.0% (+19.2pp vs. baseline). Precision and recall
  are 1.00 for both classes.
- Feature importance now splits between the two label inputs:
  obstacle_free_width_float (42.4%) and curb_height_max (37.6%),
  with curb_height_missing at 8.6%.

### What this means:
- 100% is expected here, not impressive. The label IS a rule built
  from width + curb height. Once the model sees both inputs, it just
  reconstructs the rule perfectly. Before, it only saw width, so it
  had to guess the curb half — that's where the 0.55 precision came
  from.
- Say it this way in the report: "the model perfectly recovers our
  rule-based label", not "the model achieves 100% accuracy". The
  first is honest. The second oversells a label we defined ourselves.
- The real takeaway: this confirms our pipeline is consistent
  end-to-end. The interesting modeling result in this project is
  Housing's null finding, not PMR's 100%.
- If we want a PMR model that predicts something it wasn't given, a
  next step would be predicting the label from features OUTSIDE the
  rule (crossing, length, width_fill only) — that would measure how
  well other attributes proxy for accessibility.

### Next step:
-> Step 4: Housing Features + Random Forest

In [14]:
# Step 4: Housing - Features + Random Forest
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

housing = pd.read_csv('../data/processed/housing_clean.csv')

housing = housing[housing['sidewalk_ok'].isin(['yes', 'no'])].copy()
housing['target'] = (housing['sidewalk_ok'] == 'yes').astype(int)

housing['state'] = housing['aadress'].str.extract(r',\s*([A-Z]{2})\s*\d{5}')
housing['state'] = housing['state'].fillna('unknown')
# NOTE: still extracting state for bias analysis. Just not feeding it
# into the model anymore.

# BEFORE: house_features = ['house_types', 'house_types:confidence',
#                            'residential_yes:confidence', 'state']
# CHANGED: dropped state. It was 89% of feature importance -- the model
# was learning geography, not house condition. This tests option 1 from
# the bias discussion: can the model still learn from real housing
# features without it?
house_features = ['house_types', 'house_types:confidence',
                   'residential_yes:confidence']

h_model_df = housing[house_features + ['target']].copy()
h_model_df['house_types'] = h_model_df['house_types'].fillna('unknown')

# CHANGED: name both encoders so Step 8 can save them
house_types_encoder = LabelEncoder()
h_model_df['house_types'] = house_types_encoder.fit_transform(h_model_df['house_types'])

# BEFORE: state_encoder = LabelEncoder()
#         h_model_df['state'] = state_encoder.fit_transform(h_model_df['state'])
# CHANGED: removed. state isn't in house_features anymore, so encoding
# it just wastes a step and confuses Step 8's export.

h_model_df = h_model_df.fillna(h_model_df.median(numeric_only=True))

X = h_model_df[house_features]
y = h_model_df['target']
X_train_house, X_test_house, y_train_house, y_test_house = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_housing = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced', random_state=42
)
rf_housing.fit(X_train_house, y_train_house)
housing_pred = rf_housing.predict(X_test_house)

acc = accuracy_score(y_test_house, housing_pred)
baseline = y_test_house.value_counts(normalize=True).max()

print("Housing Random Forest")
print(f"Accuracy:                {acc:.1%}")
print(f"Majority-class baseline: {baseline:.1%}  (always guessing 'ok')")
print(f"Gap vs. baseline:        {(acc - baseline)*100:+.1f} percentage points "
      f"({'below' if acc < baseline else 'above'} baseline)")
print()
print(classification_report(y_test_house, housing_pred, target_names=['not_ok', 'ok']))

housing_importance = pd.Series(rf_housing.feature_importances_, index=house_features).sort_values(ascending=False)
print("\nFeature importance (higher = more influence on the prediction):")
for feat, score in housing_importance.items():
    print(f"  {feat:<28} {score:.1%}")


Housing Random Forest
Accuracy:                85.0%
Majority-class baseline: 85.3%  (always guessing 'ok')
Gap vs. baseline:        -0.3 percentage points (below baseline)

              precision    recall  f1-score   support

      not_ok       0.17      0.01      0.01       189
          ok       0.85      1.00      0.92      1096

    accuracy                           0.85      1285
   macro avg       0.51      0.50      0.46      1285
weighted avg       0.75      0.85      0.79      1285


Feature importance (higher = more influence on the prediction):
  house_types:confidence       41.5%
  residential_yes:confidence   40.9%
  house_types                  17.6%


## Step 4 Summary - Housing Random Forest

### What we did:
- Filtered to rows with a real `sidewalk_ok` label. Still extract
  `state` from the address for the bias write-up, but dropped it from
  the model features (option 1 from the bias discussion). Trained on
  `house_types`, `house_types:confidence`,
  `residential_yes:confidence` only. Same settings as PMR's model.

### What we found:
- Accuracy 85.0%, basically tied with the 85.3% baseline (-0.3pp).
  Dropping state did not reveal a hidden house-condition signal.
- not_ok recall fell to 0.01, precision to 0.17. The model now almost
  always predicts "ok". Same as guessing the majority class. The old
  state-included version at least caught some bad sidewalks
  (recall 0.42).
- Top features are now house_types:confidence (41.5%),
  residential_yes:confidence (40.9%), house_types (17.6%). None of
  them separate ok from not_ok well.

### What this means:
- This confirms the diagnosis. It's a feature problem, not a tuning
  problem. state wasn't just adding bias -- it was the only feature
  with real signal, even if that signal was geography.
- We checked the raw CSV for other usable columns (house age, repair
  history, anything). None exist. house_types, the two confidence
  columns, and state/location are all this dataset has.
- Honest conclusion for the report: this dataset can't predict
  sidewalk condition from house-level features alone. That's a data
  limitation, not a modeling failure. Say it directly.

### Next step:
-> Step 5: Compare Both Random Forest Models


In [15]:
# Step 5: Compare Both Random Forest Models
# Side-by-side comparison. This is where "combining" the two
# datasets happens -- comparing results, not merging rows.
# Run after Steps 3 and 4 in the same kernel session.
import pandas as pd

pmr_acc = accuracy_score(y_test_pmr, pmr_pred)
pmr_baseline = y_test_pmr.value_counts(normalize=True).max()
house_acc = accuracy_score(y_test_house, housing_pred)
house_baseline = y_test_house.value_counts(normalize=True).max()

comparison = pd.DataFrame({
    'PMR (pmr_accessible)': {
        'Accuracy': f"{pmr_acc:.1%}",
        'Majority baseline': f"{pmr_baseline:.1%}",
        'Gap vs. baseline': f"{(pmr_acc - pmr_baseline)*100:+.1f}pp",
        'Top feature': pmr_importance.idxmax(),
        'Top feature importance': f"{pmr_importance.max():.1%}",
    },
    'Housing (sidewalk_ok)': {
        'Accuracy': f"{house_acc:.1%}",
        'Majority baseline': f"{house_baseline:.1%}",
        'Gap vs. baseline': f"{(house_acc - house_baseline)*100:+.1f}pp",
        'Top feature': housing_importance.idxmax(),
        'Top feature importance': f"{housing_importance.max():.1%}",
    },
})

print(comparison)

                             PMR (pmr_accessible)   Housing (sidewalk_ok)
Accuracy                                   100.0%                   85.0%
Majority baseline                           80.8%                   85.3%
Gap vs. baseline                          +19.2pp                  -0.3pp
Top feature             obstacle_free_width_float  house_types:confidence
Top feature importance                      42.4%                   41.5%


## Step 5 Summary - Comparing the Two Models

### What we did:
- Put PMR's and Housing's accuracy, baseline, and top feature
  side-by-side in one table.

### What we found:
- PMR: 100.0% vs. 80.8% baseline (+19.2pp). Housing, after dropping
  state: 85.0% vs. 85.3% baseline (-0.3pp), with not_ok recall near
  zero (0.01). It's a majority-class guesser now.
- PMR's top features are the two label inputs:
  obstacle_free_width_float (42.4%) and curb_height_max (37.6%).
  Housing's top feature is house_types:confidence (41.5%) -- not
  geography anymore, but not actually predictive either.

### What this means:
- Neither number should be read at face value. PMR's 100% isn't
  model skill -- the model now sees both inputs of our own label rule
  (width + curb height), so it just reconstructs the rule. Housing's
  85% isn't skill either -- it's the majority-class rate.
- The honest summary of both models: PMR confirms our pipeline is
  consistent end to end. Housing is a null finding -- once the biased
  feature (state) is removed, there isn't enough signal left to beat
  the baseline. That's a data limitation worth reporting, not
  something to hide behind a different algorithm.
- The two datasets still don't transfer to each other. Separate
  pipelines stays the right call.

### Next step:
-> Step 6: ModernBERT Embeddings (Housing address text)

In [16]:
# Step 6: ModernBERT Embeddings (Housing address text)
# Embed Housing's aadress field. PMR has no free text field worth
# embedding (just categories and numbers), so this step is
# Housing-only.
#
# NOTE: this cell downloads a model from Hugging Face the first time
# you run it. Needs internet. Run this locally or in Colab, not in a
# sandbox without internet.
# pip install sentence-transformers  (run once, then restart kernel)
import pandas as pd
import numpy as np
import time
from sentence_transformers import SentenceTransformer

housing = pd.read_csv('../data/processed/housing_clean.csv')
housing = housing[housing['sidewalk_ok'].isin(['yes', 'no'])].copy()

# nomic-ai/modernbert-embed-base needs a "search_document:" prefix on
# the text you embed (see the model card).
addresses = ('search_document: ' + housing['aadress'].astype(str)).tolist()

model = SentenceTransformer('nomic-ai/modernbert-embed-base')

# CHANGED: time the encoding step so we know how long this takes
start = time.time()
address_embeddings = model.encode(addresses, show_progress_bar=True)
elapsed = time.time() - start

# CHANGED: readable, labeled output instead of a raw shape tuple
n_addresses, n_dims = address_embeddings.shape
print(f"Embedded {n_addresses:,} addresses into {n_dims}-dimensional vectors")
print(f"Time taken: {elapsed:.1f} seconds ({n_addresses/elapsed:.0f} addresses/sec)")

np.save('../data/processed/housing_address_embeddings.npy', address_embeddings)
print(f"Saved to: ../data/processed/housing_address_embeddings.npy")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Batches:   0%|          | 0/201 [00:00<?, ?it/s]

Embedded 6,425 addresses into 768-dimensional vectors
Time taken: 137.8 seconds (47 addresses/sec)
Saved to: ../data/processed/housing_address_embeddings.npy


## Step 6 Summary - ModernBERT Embeddings

### What we did:
- Embedded Housing's `aadress` text with
  `nomic-ai/modernbert-embed-base`. One vector per row.
- Skipped PMR. Its fields are categories and numbers, no free text
  worth embedding.

### What we found:
- Embedded 6,425 addresses into 768-dimensional vectors.
- Took 137.8 seconds (about 47 addresses/second) on this run.

### What this means:
- These vectors capture rough address similarity (same city, similar
  street names, etc). They feed into Step 7's FAISS index.

### Next step:
-> Step 7: FAISS Similarity Search


In [17]:
# Step 7: FAISS Similarity Search
# Build one FAISS index per dataset. Search stays within each
# dataset -- PMR to PMR, Housing to Housing -- since they don't
# share geography (see intro note).
import numpy as np
import pandas as pd
import faiss
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 7.1 Housing: search over the ModernBERT embeddings from Step 6.
# ---------------------------------------------------------
address_embeddings = np.load('../data/processed/housing_address_embeddings.npy').astype('float32')
housing_index = faiss.IndexFlatL2(address_embeddings.shape[1])
housing_index.add(address_embeddings)

# Example: find the 5 addresses most like row 0.
distances, neighbor_ids = housing_index.search(address_embeddings[:1], 5)

query_address = housing['aadress'].iloc[0]
print(f"Housing: addresses most similar to row 0 ({query_address}):")
for rank, (nid, dist) in enumerate(zip(neighbor_ids[0], distances[0]), start=1):
    print(f"  #{rank}  row {nid:<6} distance={dist:.3f}  {housing['aadress'].iloc[nid]}")

faiss.write_index(housing_index, '../data/processed/housing_faiss.index')

# ---------------------------------------------------------
# 7.2 PMR: no text to embed, so build the index over numeric features
# instead. Still useful -- e.g. "find other segments with a similar
# width/curb/crossing profile to this bad one".
#
# BEFORE: this section added curb_height_max and curb_height_missing
# here, on top of the 4 features from Step 3.
# CHANGED: Step 3 already adds both now (to fix low precision -- see
# Step 3). Adding them again here would duplicate the columns. So
# retrieval just reuses pmr_features. NaNs were already filled by
# Step 3's fillna(median).
# ---------------------------------------------------------
pmr_retrieval_features = pmr_features

scaler = StandardScaler()
pmr_vectors = scaler.fit_transform(
    pmr_model_df[pmr_retrieval_features].to_numpy()
).astype('float32')

pmr_index = faiss.IndexFlatL2(pmr_vectors.shape[1])
pmr_index.add(pmr_vectors)

# Query on a real "not accessible" segment, not row 0 (which happens
# to be one of many exact-duplicate rows -- see Step 7 Summary).
query_idx = pmr_model_df[pmr_model_df['pmr_accessible'] == 0].index[0]
query_vector = pmr_vectors[query_idx:query_idx + 1]
distances, neighbor_ids = pmr_index.search(query_vector, 5)

print(f"\nPMR: segments most similar to row {query_idx} (a 'not accessible' segment):")
print(f"  Query row {query_idx} features:")
for feat in pmr_retrieval_features:
    val = pmr_model_df.loc[query_idx, feat]
    print(f"    {feat:<28} {val}")
for rank, (nid, dist) in enumerate(zip(neighbor_ids[0], distances[0]), start=1):
    tag = " (identical values)" if dist == 0 else ""
    same_label = pmr_model_df.loc[nid, 'pmr_accessible']
    print(f"  #{rank}  row {nid:<6} distance={dist:.3f}  "
          f"pmr_accessible={same_label}{tag}")

faiss.write_index(pmr_index, '../data/processed/pmr_faiss.index')


Housing: addresses most similar to row 0 (4210 Southwest 167th Avenue, Beaverton, OR 97007, USA ):
  #1  row 0      distance=0.000  4210 Southwest 167th Avenue, Beaverton, OR 97007, USA 
  #2  row 3775   distance=0.154  8350 Southwest 168th Avenue, Beaverton, OR 97007, USA
  #3  row 1027   distance=0.268  6360 Southwest 154th Place, Beaverton, OR 97007, USA
  #4  row 4550   distance=0.461  15870 Southwest Bridle Hills Drive, Beaverton, OR 97007, USA
  #5  row 6309   distance=0.464  9350 Southwest Galena Way, Beaverton, OR 97007, USA

PMR: segments most similar to row 145 (a 'not accessible' segment):
  Query row 145 features:
    length                       1.99
    obstacle_free_width_float    0.8
    crossing                     0
    width_fill                   0.0
    curb_height_max              0.06
    curb_height_missing          1
  #1  row 145    distance=0.000  pmr_accessible=0 (identical values)
  #2  row 199    distance=0.000  pmr_accessible=0 (identical values)
  #3  ro

## Step 7 Summary - FAISS Similarity Search

### What we did:
- Built one FAISS index for Housing (ModernBERT address embeddings)
  and one for PMR (numeric features, PMR has no text).
- For PMR, reused the same 6 features from Step 3 (curb height +
  missing flag included). Standardized before indexing.
- Both indexes stay inside their own dataset. No cross-dataset
  lookups. Same reason we don't merge PMR and Housing.

### What we found:
- Housing: row 0 (Beaverton, OR) returned other Beaverton addresses.
  Distances go up smoothly, 0.154 to 0.464. Ranking works.
- PMR: all 5 neighbors are exact ties (distance = 0.000). Adding
  curb_height_max didn't help. 74% of rows have no curb height, they
  all get the same fill value. The other features repeat a lot too.
  Many segments have identical values. Only location differs, and
  location isn't in this index.
- Our query row shows the problem: its curb_height_max is 0.06, but
  that's just the fill value (curb_height_missing = 1).

### What this means:
- Housing index: "find addresses like this one." Good for flagging
  nearby addresses once one is marked not accessible.
- PMR index: a group finder, not a ranker. It groups segments with
  the same profile (all 5 matches really are pmr_accessible=0). It
  can't rank inside the group. This is a data limit, not a code bug.
  Picking another demo row won't fix it. More features of the same
  kind won't fix it. A real fix needs coordinates in the index, but
  then it's location search, not attribute search.

### Next step:
-> Step 8: Export Models, Embeddings, Index

In [18]:
# Step 8: Export Models, Embeddings, Index
import os
import joblib

os.makedirs('../data/models', exist_ok=True)

# Random Forest models
joblib.dump(rf_pmr, '../data/models/rf_pmr.pkl')
joblib.dump(rf_housing, '../data/models/rf_housing.pkl')

# Also save the encoders/scaler used to build model inputs.
# Without these, new data can't be transformed the same way at
# inference time.
joblib.dump(crossing_encoder, '../data/models/crossing_encoder.pkl')
joblib.dump(house_types_encoder, '../data/models/house_types_encoder.pkl')
# BEFORE: joblib.dump(state_encoder, '../data/models/state_encoder.pkl')
# CHANGED: removed. Step 4 stopped encoding state, so state_encoder
# doesn't exist anymore. Keeping this line would throw a NameError.
joblib.dump(scaler, '../data/models/pmr_retrieval_scaler.pkl')

# Grouped by category for a more readable summary
saved_files = {
    "Models": [
        ('../data/models/rf_pmr.pkl', 'PMR Random Forest model'),
        ('../data/models/rf_housing.pkl', 'Housing Random Forest model'),
    ],
    "Encoders & Scalers": [
        ('../data/models/crossing_encoder.pkl', 'PMR crossing LabelEncoder'),
        ('../data/models/house_types_encoder.pkl', 'Housing house_types LabelEncoder'),
        # BEFORE: state_encoder.pkl was listed here. Removed with the save above.
        ('../data/models/pmr_retrieval_scaler.pkl', 'PMR retrieval StandardScaler'),
    ],
    "Embeddings & Indexes": [
        ('../data/processed/housing_address_embeddings.npy', 'Housing ModernBERT embeddings (Step 6)'),
        ('../data/processed/housing_faiss.index', 'Housing FAISS index (Step 7)'),
        ('../data/processed/pmr_faiss.index', 'PMR FAISS index (Step 7)'),
    ],
}

n_total = sum(len(v) for v in saved_files.values())
n_missing = sum(
    1 for group in saved_files.values() for path, _ in group if not os.path.exists(path)
)

print("*" * 100)
print("STEP 8 EXPORT SUMMARY")
print("*" * 100)
for group_name, files in saved_files.items():
    print(f"\n{group_name}:")
    for path, desc in files:
        status = "OK" if os.path.exists(path) else "MISSING"
        print(f"  [{status}] {path:<45} {desc}")
print("-" * 100)
print(f"Total files: {n_total}   |   Missing: {n_missing}")
print("*" * 100)


****************************************************************************************************
STEP 8 EXPORT SUMMARY
****************************************************************************************************

Models:
  [OK] ../data/models/rf_pmr.pkl                     PMR Random Forest model
  [OK] ../data/models/rf_housing.pkl                 Housing Random Forest model

Encoders & Scalers:
  [OK] ../data/models/crossing_encoder.pkl           PMR crossing LabelEncoder
  [OK] ../data/models/house_types_encoder.pkl        Housing house_types LabelEncoder
  [OK] ../data/models/pmr_retrieval_scaler.pkl       PMR retrieval StandardScaler

Embeddings & Indexes:
  [OK] ../data/processed/housing_address_embeddings.npy Housing ModernBERT embeddings (Step 6)
  [OK] ../data/processed/housing_faiss.index         Housing FAISS index (Step 7)
  [OK] ../data/processed/pmr_faiss.index             PMR FAISS index (Step 7)
---------------------------------------------------------------

## Step 8 Summary - Export Models, Embeddings, Index

**What we did:**
- Saved both Random Forest models (`rf_pmr.pkl`, `rf_housing.pkl`) to `data/models/`.
- Saved the encoders/scaler used to build model inputs (`crossing_encoder.pkl`, `house_types_encoder.pkl`, `pmr_retrieval_scaler.pkl`), so new data can be transformed the same way at inference time.
- Dropped `state_encoder.pkl` from the export. Step 4 stopped encoding state (bias fix), so there's nothing to save.
- Embeddings and FAISS indexes were already saved in Steps 6-7; this step just verifies and reports on them alongside everything else.
- Grouped the summary into 3 categories with a file check per line. A broken run shows MISSING right away instead of failing later.

**What we found:**
- All 8 artifacts saved (was 9 before dropping state_encoder): 2 models, 3 encoders/scaler, 1 embeddings file, 2 FAISS indexes.
- 0 missing files across all three categories.

**What this means:**
- The pipeline runs end to end. Everything downstream (a demo app, further analysis, or scoring new addresses/segments) can load these files instead of retraining from scratch.
- The Housing model no longer needs a state value at inference time. One less field a new address needs.

**Next step:**
→ Pipeline is done. What's left: bias write-up, README, and the report.